***
### Main Idea:
A single global vector:

IPV=[v0,v1,v2,v3,v4]

that tells SRRIP:

how aggressively to insert new cache lines
based on OPT-derived behavior

### Key assumption:

Can OPT’s behavior be compressed into a single insertion bias vector?

we are NOT learning:
  - per-program policy
  - per-address policy
  - ML features

You are learning:
  - a universal cache insertion priorirty

### Finding IPV: 
* Run OPT on traces
  - You simulate ideal cache behavior
    - input: memory traces
    - output: exact eviction decisions + lifetimes

  - This gives you:
    - how long each cache line should have lived

1. Need to define reuse-distance buckets
  - Convert reuse distance → classes:

    Example:

    - D = 0–10        → very hot
    - D = 10–100      → warm
    - D = 100–1K      → cold
    - D = >1K         → dead-on-arrival

2. Observe OPT insertion outcomes
  - For every insertion track:
    - Was the block reused before eviction?
    - How long did it stay?
    - What was its reuse distance?

3. Map to IPV values
  - IPV should reflect expected usefulness at insertion
    - High reuse probability → high IPV (insert near MRU)
    - Low reuse probability → low IPV (insert near LRU)

  Example mapping:

  - Bucket            → rrpv
  - very hot          → 0
  - warm              → 1
  - cold              → 2
  - dead              → 3

* Predict probability of survival until next access
  - IPV = P(survive)

* Update on Hit
  - How to treat hits if IPV does not model state transitions.
  - RRPV_new = (1-alpha)RRPV_old + a*EIPV
  - EIPV = Sum( i * P_instr/data(i) )
  - Larger alpha = faster movement toward 0.
  - Smaller alpha = more inertia/history.


***
Compare LFU, PACIPV (Best vector with exhaustive search), Belady-driven-sampling
- tuning alphas
 

In [ ]:
# Data collection for Belady's sweep. Run this before running the notebook to plot the results.
python3 ./scripts/run_belady_sweep.py  \
  --train-workloads selected_qualcomm_srv_ap \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --train-fractions 1.0  \
  --policies pacipv,lfu,belady_driven_sampling   \
  --pacipv-output-file-template 'data/belady_sweep/pacipv_{train_workload}_to_{eval_workload}_s{num_sets}_w{num_ways}_frac{train_fraction_tag}_alpha{bds_alpha_tag}.txt' \
  --bds-alphas 1.0,0.7,0.5,0.2,0 \
  --slurm

In [ ]:
# Plotting the results after running the above data collection.
python3 ./scripts/plot_belady_sweep_results.py \
  --sweep-summary-csv ./data/belady_sweep/belady_sweep_summary.csv \
  --train-workloads selected_qualcomm_srv_ap \
  --eval-workloads selected_qualcomm_srv_ap \
  --num-sets-list 64 --num-ways-list 16 \
  --train-fractions 1.0 \
  --bds-alphas 1.0,0.7,0.5,0.2,0 \
  --policies belady_driven_sampling \
  --summary-csv data/belady_sweep/plot_lookup_summary.csv